# 03. Feature Engineering - Churn Predictors & Usage Aggregation
**Author:** Ravikant Yadav  
**Date:** June 23, 2026  

To build robust insights on user stickiness, we must translate granular transactional logs and activity events into user-level features. In this notebook, we engineer features like:
- Customer Lifespan/Tenure (months)
- Granular platform usage intensity (events & active minutes)
- Support ticket volume & average SLA resolution times
- Payment failure rates


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "cleaned"

subscriptions = pd.read_csv(DATA_DIR / "subscriptions.csv")
users = pd.read_csv(DATA_DIR / "users.csv")
product_usage = pd.read_csv(DATA_DIR / "product_usage.csv")
support_tickets = pd.read_csv(DATA_DIR / "support_tickets.csv")
payments = pd.read_csv(DATA_DIR / "payments.csv")


## 1. Feature 1: Subscription Tenure (Months)
Let's calculate how many months a subscriber has been with the company, using their start date and end date (or our baseline date `2026-01-01` for active users).


In [ ]:
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])
baseline_date = pd.to_datetime('2026-01-01')

tenure_months = []
for idx, row in subscriptions.iterrows():
    end = row['end_date'] if pd.notnull(row['end_date']) else baseline_date
    delta = end - row['start_date']
    # Approximate months
    months = delta.days / 30.4368
    tenure_months.append(max(0.1, round(months, 1)))

subscriptions['tenure_months'] = tenure_months
print(subscriptions[['subscription_id', 'start_date', 'status', 'tenure_months']].head())


## 2. Feature 2: Product Usage Intensity (Events & Active Minutes)
Let's aggregate product usage event counts and total active minutes per user from `product_usage.csv`.


In [ ]:
print("Product usage columns:", product_usage.columns.tolist())

# Aggregate by user_id
usage_agg = product_usage.groupby('user_id').agg(
    total_features_used=('feature', 'nunique'),
    total_usage_events=('events', 'sum'),
    total_active_minutes=('active_minutes', 'sum'),
    avg_active_minutes_per_event=('active_minutes', 'mean')
).reset_index()

print("--- Engineered Product Usage Features ---")
print(usage_agg.head())


## 3. Feature 3: Customer Support Ticketing Severity
Let's calculate total tickets raised and average resolution times for each user, as support friction is highly correlated with subscription churn.


In [ ]:
# Aggregate support tickets
ticket_agg = support_tickets.groupby('user_id').agg(
    total_support_tickets=('ticket_id', 'count'),
    avg_first_response_hours=('first_response_hours', 'mean'),
    avg_resolution_hours=('resolution_hours', 'mean')
).reset_index()

print("--- Engineered Customer Support Features ---")
print(ticket_agg.head())


## 4. Feature 4: Payment Friction (Decline Rate)
Let's calculate the percentage of failed billing attempts for each user from our payment logs.


In [ ]:
payment_agg = payments.groupby('user_id').agg(
    total_payment_attempts=('payment_id', 'count'),
    failed_payment_attempts=('payment_status', lambda x: (x == 'Failed').sum())
).reset_index()

payment_agg['payment_failure_rate'] = (payment_agg['failed_payment_attempts'] / payment_agg['total_payment_attempts']).fillna(0)
print("--- Engineered Payment Friction Features ---")
print(payment_agg.head())


## 5. Build Master Feature Matrix
Now, let's merge these user-level features into a single master feature dataset. This consolidated matrix gives us a unified view of each customer, combining account age, platform engagement, support issues, and payment friction.


In [ ]:
# Merge subscriptions, users, usage, support, and payments
master_df = subscriptions[['user_id', 'subscription_id', 'plan_id', 'monthly_price', 'status', 'tenure_months']].copy()
master_df = pd.merge(master_df, users[['user_id', 'company_size', 'region', 'acquisition_channel']], on='user_id', how='left')
master_df = pd.merge(master_df, usage_agg, on='user_id', how='left')
master_df = pd.merge(master_df, ticket_agg, on='user_id', how='left')
master_df = pd.merge(master_df, payment_agg, on='user_id', how='left')

# Fill NaNs with sensible defaults
master_df['total_features_used'] = master_df['total_features_used'].fillna(0)
master_df['total_usage_events'] = master_df['total_usage_events'].fillna(0)
master_df['total_active_minutes'] = master_df['total_active_minutes'].fillna(0)
master_df['avg_active_minutes_per_event'] = master_df['avg_active_minutes_per_event'].fillna(0)
master_df['total_support_tickets'] = master_df['total_support_tickets'].fillna(0)
master_df['avg_first_response_hours'] = master_df['avg_first_response_hours'].fillna(24.0) # Assume slow response if no ticket
master_df['avg_resolution_hours'] = master_df['avg_resolution_hours'].fillna(48.0) # Assume slow resolution if no ticket
master_df['total_payment_attempts'] = master_df['total_payment_attempts'].fillna(0)
master_df['failed_payment_attempts'] = master_df['failed_payment_attempts'].fillna(0)
master_df['payment_failure_rate'] = master_df['payment_failure_rate'].fillna(0.0)

# Create binary target variable: is_churned
master_df['is_churned'] = master_df['status'].apply(lambda x: 1 if x == 'Churned' else 0)

print("\n--- Master Customer Feature Matrix Completed ---")
print(master_df.info())
print(master_df.describe().T)
